<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Synthetic-Data---ARMA(1,1)" data-toc-modified-id="Synthetic-Data---ARMA(1,1)-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Synthetic Data - ARMA(1,1)</a></span></li><li><span><a href="#Statsmodels" data-toc-modified-id="Statsmodels-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Statsmodels</a></span></li><li><span><a href="#PyMC" data-toc-modified-id="PyMC-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>PyMC</a></span><ul class="toc-item"><li><span><a href="#Prior-Predictive" data-toc-modified-id="Prior-Predictive-3.1"><span class="toc-item-num">3.1&nbsp;&nbsp;</span>Prior Predictive</a></span><ul class="toc-item"><li><span><a href="#Unconditional" data-toc-modified-id="Unconditional-3.1.1"><span class="toc-item-num">3.1.1&nbsp;&nbsp;</span>Unconditional</a></span></li><li><span><a href="#Conditional" data-toc-modified-id="Conditional-3.1.2"><span class="toc-item-num">3.1.2&nbsp;&nbsp;</span>Conditional</a></span></li></ul></li></ul></li><li><span><a href="#Estimation" data-toc-modified-id="Estimation-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Estimation</a></span></li><li><span><a href="#Posterior-Predictive" data-toc-modified-id="Posterior-Predictive-5"><span class="toc-item-num">5&nbsp;&nbsp;</span>Posterior Predictive</a></span><ul class="toc-item"><li><span><a href="#Unconditional" data-toc-modified-id="Unconditional-5.1"><span class="toc-item-num">5.1&nbsp;&nbsp;</span>Unconditional</a></span></li><li><span><a href="#Conditional" data-toc-modified-id="Conditional-5.2"><span class="toc-item-num">5.2&nbsp;&nbsp;</span>Conditional</a></span></li></ul></li><li><span><a href="#Predictions-Compared" data-toc-modified-id="Predictions-Compared-6"><span class="toc-item-num">6&nbsp;&nbsp;</span>Predictions Compared</a></span></li></ul></div>

In [ ]:
%matplotlib inline


import warnings

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import statsmodels.api as sm

from pymc.model.transform.optimization import freeze_dims_and_data
from pytensor import tensor as pt

import pymc_extras.statespace as pmss

warnings.filterwarnings(action="ignore", message="The RandomType SharedVariables")


config = {
    "figure.figsize": [12.0, 4.0],
    "figure.dpi": 72.0 * 2,
    "figure.facecolor": "w",
    "figure.constrained_layout.use": True,
    "axes.grid": True,
    "grid.linewidth": 0.5,
    "grid.linestyle": "--",
    "axes.spines.top": False,
    "axes.spines.bottom": False,
    "axes.spines.left": False,
    "axes.spines.right": False,
}

plt.rcParams.update(config)


def sample_kwargs():
    return {
        "nuts_sampler": "nutpie",
        "cores": 8,
        "draws": 500,
        "tune": 1000,
    }

# Synthetic Data - ARMA(1,1)

Start very simple because I don't want to worry about stationarity at higher orders of p,q

In [ ]:
seed = sum(map(ord, "statespace arima"))
rng = np.random.default_rng(seed)

AR_params = [0.8, 0.0]
MA_params = [-0.5]

# Initial state
x0 = np.r_[[0.0], [0.0]]

# Hidden state transition matrix
T = np.array([[AR_params[0], 1.0], [AR_params[1], 0.0]])

# Hidden state noise coefficients
R = np.array([[1.0], [MA_params[0]]])

# Hidden state covariance matrix
Q = np.array([[0.8]])

# Observation matrix
Z = np.array([[1.0, 0.0]])

# Observation noise covariance
H = np.array([[0.0]])

timesteps = 100
data = np.zeros(timesteps)
hidden_states = np.zeros((timesteps, 2))
hidden_states[0, :] = x0

innovations = rng.multivariate_normal(mean=np.array([0.0]), cov=Q, size=timesteps)

for t in range(1, timesteps):
    hidden_states[t] = T @ hidden_states[t - 1, :] + R @ np.atleast_1d(innovations[t])
    data[t] = (Z @ hidden_states[t]).item()

fake_dates = pd.date_range("2010-01-01", freq="MS", periods=data.shape[0])
df = pd.DataFrame(data, columns=["state"], index=fake_dates)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4), dpi=100)
df.plot(ax=ax)
plt.show()

# Statsmodels

In [ ]:
mod = sm.tsa.SARIMAX(endog=data, order=(1, 0, 1))
res = mod.fit(disp=0)
res.summary()

# PyMC

Unlike the local level model, we don't need to explicitly estimate the inital state or covaraince matrix for an ARIMA model, **if** the model dynamics are guaranteed to be stationary. Since we have an AR(1), that means the autoregressive parameter must be less than 1 in absolute value. In this case, closed formed solutions for the steady-state of the system can be used for the initial state:

$$x^\star = (I - T)^{-1}c$$

The covariance matrix is more complex, but can be obtained by solving this quadratic matrix equation (known as a "discrete lyapunov equation"):

$$TP^{\star} T^T - P^{\star} + R Q R^T = 0$$

In [ ]:
ss_mod = pmss.BayesianSARIMAX(order=(1, 0, 1), verbose=True)

In [ ]:
ss_mod.coords

In [ ]:
ss_mod.param_dims

In [ ]:
with pm.Model(coords=ss_mod.coords) as arma_model:
    state_sigmas = pm.Gamma(
        "sigma_state", alpha=10, beta=2, dims=ss_mod.param_dims.get("sigma_state")
    )
    rho = pm.Beta("ar_params", alpha=5, beta=1, dims=ss_mod.param_dims.get("ar_params"))
    theta = pm.Normal("ma_params", mu=0.0, sigma=0.5, dims=ss_mod.param_dims.get("ma_params"))

    ss_mod.build_statespace_graph(df)

with freeze_dims_and_data(arma_model):
    prior = pm.sample_prior_predictive()

## Prior Predictive

### Unconditional

A note about what the states are, first of all. For an ARMA (1,1) model, the state space transition equation is:

$$ \begin{bmatrix} y_{t+1} \\ x_{t+1} \end{bmatrix} =
    \begin{bmatrix}\rho & 1 \\ 0 & 0 \end{bmatrix} \begin{bmatrix} y_t \\ x_t \end{bmatrix} +
    \begin{bmatrix} 1 \\ \theta \end{bmatrix} \varepsilon_{t+1}
    $$

Which, if you work out the matrix multiplications, gives two state equations:

$$\begin{align} y_{t+1} &= \rho y_t + x_t + \varepsilon_{t+1} \\
    x_{t+1} &= \theta \varepsilon_{t+1} \end{align}$$

So the two states we end up with are the modeled data ($y_{t+1}$), and the innovation series, scaled by $\theta$. To recover the estimated innovations, you would need to divide the second state by $\theta$. It is also possible to write it such that the innovations are "carried along", but it doubles the number of states required. This is probably fine for small values of p and q, but bad for larger ones, since the Kalman filter needs to invert a matrix of size $k \times k$ each iteration.

Anyway, we see can two different types of trajectories in the priors. One is sharp and jagged, when rho/theta are large and negative. The other is smooth and meandering, when rho is positive and reasonable. When rho is very close to 1, we get the trajectories that wander out to +/- 20.

In [ ]:
unconditional_prior = ss_mod.sample_unconditional_prior(prior, progressbar=False)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 6), dpi=144)
for idx, (axis, state) in enumerate(zip(fig.axes, ss_mod.state_names)):
    unconditional_prior.prior_latent.sel(state=state).stack(sample=["chain", "draw"]).plot.line(
        x="time", ax=axis, add_legend=False
    )
    axis.set(title=state)

fig.set_facecolor("w")
fig.tight_layout()
plt.show()

### Conditional

The conditional outputs for the kalman smoother and filter aren't as interesting for this model, because the process noise doesn't have full rank. That is, only the second state, associated with the MA process, has noise added. The result is that the first state noiselessly encodes the data. Let's have a look!

In [ ]:
conditional_prior = ss_mod.sample_conditional_prior(prior)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 6), dpi=144)
for idx, (axis, state) in enumerate(zip(fig.axes, ss_mod.state_names)):
    conditional_prior.filtered_prior.sel(state=state).stack(sample=["chain", "draw"]).plot.line(
        x="time", ax=axis, add_legend=False
    )
    axis.set(title=state)

fig.set_facecolor("w")
fig.tight_layout()
plt.show()

We can ask for the kalman predictions, $\mathbb E\left [ x_{t+1} | \{y_\tau\}_{\tau=0}^{t} \right]$, which are a bit more interesting, by passing `filter_output = 'predicted'` to the `sample_conditional_prior` method

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 6), dpi=144)
for idx, (axis, state) in enumerate(zip(fig.axes, ss_mod.state_names)):
    conditional_prior.predicted_prior.sel(state=state).stack(sample=["chain", "draw"]).plot.line(
        x="time", ax=axis, add_legend=False
    )
    axis.set(title=state)

fig.set_facecolor("w")
fig.tight_layout()
plt.show()

# Estimation

Finally on to actually fitting the model. There are ways to speed things up a bit. We are using stationary covariance initialization, which saves us from estimating the initial covariance matrix. PyMC Statespace also offers different flavors of Kalman Filter, some with some built in speedups (`steady_state`, but it's not JAX compatible).

For now we'll just go with the standard filter. There is a separate notebook explaining and comparing the different filtering choices.

In [ ]:
with arma_model:
    idata = pm.sample(**sample_kwargs())

In [ ]:
az.plot_trace_dist(idata, var_names=list(ss_mod.param_names))

Posterior estimates are reasonable, and the true value is contained in the 94% HDI in each case. Looks like the ML estimator did a bit better, but we also estimated 4 more parameters than it did (2 initial states, plus 2 initial covariances!)

In [ ]:
pc = az.plot_dist(idata, var_names=["sigma_state", "ma_params", "ar_params"])
for var, ml, true_val in zip(
    ["sigma_state", "ma_params", "ar_params"], res.params[:3], [0.8, -0.5, 0.8]
):
    target = pc.get_target(var, {})
    target.axvline(ml, color="tab:red", lw=2, label="ML Estimate")
    target.axvline(true_val, color="tab:green", lw=2, label="True Value")
fig.legend()
plt.show()

# Posterior Predictive

## Unconditional

In [ ]:
unconditional_post = ss_mod.sample_unconditional_posterior(
    idata, steps=100, use_data_time_dim=False
)
post = az.extract(unconditional_post, "posterior_predictive")

Notice how there's no more trajectories that diverge away from the mean. The posterior rules out values of $\rho$ close to 1. Since we ended up with a stationary model, the long-term unconditional dynamics converge to a steady state mean and covariance matrix, which is what we are seeing here.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 6), dpi=144)
hdi = az.hdi(unconditional_post.to_dataset().posterior_latent)

for idx, (axis, name) in enumerate(
    zip(fig.axes, ["Observed State (Data)", "Hidden State (ARMA Dynamics)"])
):
    post.isel(state=idx).mean(dim="sample").posterior_latent.plot.line(
        x="time", ax=axis, color="k", lw=2, add_legend=False
    )
    axis.fill_between(hdi.coords["time"], *hdi.isel(state=idx).values.T, alpha=0.25)
    axis.set(title=name)

fig.set_facecolor("w")
fig.tight_layout()
plt.show()

As a model check, we can compute the closed-form steady state distributon of the statespace system under the posterior and check whether the simulations are convering to it. The closed form mean and covariance matrix are:

$$\begin{align}
\mu^\star &= (I - T)^{-1} c \\
\Sigma^\star &= T \Sigma^\star T^T − T \Sigma^\star Z^T F{-1} Z \Sigma^\star T^T + R Q R^T
\end{align}$$

The first equation assumes that $T$ is invertable, which means that the system needs to be stationary. These results thus don't apply to, say, a Gaussian Random Walk model. The second equation is a Discrete Matrix Riccati Equation. A `pytensor` solver is available, but is not JAX compatible.

In [ ]:
with pm.Model(coords=ss_mod.coords):
    ss_mod._build_dummy_graph()
    ss_mod._insert_random_variables()
    x0, P0, c, d, T, Z, R, H, Q = ss_mod.unpack_statespace()

    mu_star = pm.Deterministic(
        "mu_star", pt.linalg.inv(pt.eye(ss_mod.k_states) - T) @ c, dims=["state"]
    )
    Sigma_star = pm.Deterministic(
        "Sigma_star",
        pt.linalg.solve_discrete_are(T.T, Z.T, R @ Q @ R.T, H),
        dims=["state", "state_aux"],
    )

    idata_steady = pm.sample_posterior_predictive(
        idata, var_names=["mu_star", "Sigma_star"], random_seed=1337
    )

After 100 time steps, it looks like the covariance of the samples is approaching the analyical posterior covariance.

In [ ]:
np.cov(post.posterior_latent.isel(time=-1).values)

In [ ]:
idata_steady.posterior_predictive.mean(dim=["chain", "draw"]).Sigma_star.values

## Conditional

In [ ]:
post_pred = ss_mod.sample_conditional_posterior(idata)

Comparing the posterior outputs, you can see that only the Kalman predictions have posterior errors. This is because there is no measurement error in this model, nor unobserved shocks to the hidden states. In the context of the Kalman Filter, this has the effect of noiselessly encoding the data during the filter step (and the smoother is based on the filtered distribution).

In all three posterior predicitive distributions, we can see that there is noise in the hidden state estimation. This amounts to estimation of the innovation series, scaled by the MA parameter $\theta$ (refer to the algebra above). This amounts to draws from a fixed Normal distribution in the filtered and predictive distributions, but more closely follows the data in the smoothed distribution, since this takes into account both past and future observations.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 6), dpi=144)
post = az.extract(post_pred, "posterior_predictive")
colors = ["tab:" + c for c in ["blue", "orange", "green"]]

for filter_output, color in zip(["filtered", "predicted", "smoothed"], colors):
    hdi = az.hdi(post_pred.to_dataset())[f"{filter_output}_posterior"]
    for idx, (axis, name) in enumerate(
        zip(fig.axes, ["Observed State (Data)", "Hidden State (ARMA Dynamics)"])
    ):
        post[f"{filter_output}_posterior"].isel(state=idx).mean(dim="sample").plot.line(
            x="time", ax=axis, lw=2, add_legend=False, label=filter_output.title(), color=color
        )
        axis.fill_between(
            hdi.coords["time"], *hdi.isel(state=idx).values.T, alpha=0.25, color=color
        )
        axis.set(title=name)

ax[0].legend()
fig.set_facecolor("w")
fig.tight_layout()
plt.show()

# Predictions Compared

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4), dpi=100)
hdi = az.hdi(post_pred.to_dataset()).predicted_posterior

ax.plot(df.index, res.states.predicted[:-1, 0], label="Statsmodel (ML)")
post.mean(dim="sample").isel(state=0).predicted_posterior.plot.line(
    x="time", label="Bayesian (Mean)"
)
ax.fill_between(
    df.index,
    *hdi.isel(state=0).values.T,
    alpha=0.25,
    color="tab:orange",
)

ml_mu = res.states.predicted[:-1, 0]
ml_var = res.states.predicted_cov.reshape(-1, 2, 2)[:-1, 0, 0]

ax.fill_between(
    df.index,
    ml_mu - 1.96 * np.sqrt(ml_var),
    ml_mu + 1.96 * np.sqrt(ml_var),
    alpha=0.25,
    color="tab:blue",
)

ax.plot(df.index, df.values, label="Observed")
ax.set(title="Observed State (Local Level)")
ax.legend()

fig.set(facecolor="white")
fig.tight_layout()
plt.show()

## Forecasting

Forecasting is also easy, just use the `forecast` method.

In [ ]:
def plot_forecast(df, forecast):
    fig, ax = plt.subplots(figsize=(14, 4), dpi=100)
    ax.plot(df.index, df.values, label="Data")

    hdi_forecast = az.hdi(idata_forecast.to_dataset()).forecast_observed

    forecast.mean(dim="sample").forecast_observed.isel(observed_state=0).plot.line(
        x="time", label="Forecast", add_legend=False
    )
    ax.fill_between(
        forecast.coords["time"].values,
        *hdi_forecast.isel(observed_state=0).values.T,
        alpha=0.25,
        color="tab:orange",
    )

    ax.legend()
    plt.show()

In [ ]:
idata_forecast = ss_mod.forecast(idata, start=df.index[-1], periods=10)
forecast = az.extract(idata_forecast, "posterior_predictive")
plot_forecast(df, forecast)

## Porcupine Graph

A "porcupine graph" shows model forecasts at different periods in the time series. The name comes from the fact that forecasts lines poke out from the data like porcupine quills. We make one for this data to show the flexibility of the forecast method.

This isn't a true porcupine graph though, because the "forecasts" for the in-sample period are generated using parameters fit on data from the future. As noted, it's just a nice demonstration of the forecast method.

In [ ]:
import warnings

idatas = []
for date in df.index[10::10]:
    with warnings.catch_warnings():
        # silence the PyMC warning about JAX shared variables
        warnings.simplefilter("ignore")
        idata_forecast = ss_mod.forecast(
            idata, start=date, periods=5, filter_output="smoothed", progressbar=False
        )
    idatas.append(idata_forecast)

In [ ]:
fig, ax = plt.subplots()
ax.plot(df.index, df.values, lw=2, c="k")

for idata_forecast in idatas:
    forecast = az.extract(idata_forecast, "posterior_predictive")
    hdi_forecast = az.hdi(idata_forecast.to_dataset(), prob=0.69).forecast_observed

    mean = forecast.mean(dim="sample").forecast_observed.isel(observed_state=0)
    mean.plot.line(x="time", color="tab:blue", ls="--")
    ax.fill_between(
        hdi_forecast.coords["time"].values,
        *hdi_forecast.isel(observed_state=0).values.T,
        alpha=0.25,
        color="tab:blue",
    )
ax.set_title("Porcupine Graph of 10-Period Forecasts (parameters estimated on all data)")
plt.show()

## Impulse Response Function

In [ ]:
def plot_irf(irf, title, ax=None, show=True):
    if ax is None:
        fig, ax = plt.subplots()

    mean = irf.irf.mean(dim=["chain", "draw"])
    hdi_50 = az.hdi(irf.to_dataset(), 0.5).irf
    hdi = az.hdi(irf.to_dataset()).irf

    x_time = irf.coords["time"]

    ax.plot(x_time, mean, color="k", label="Mean")
    ax.fill_between(x_time, *hdi.values.T, color="tab:blue", alpha=0.25, label="HDI 94%")
    ax.fill_between(x_time, *hdi_50.values.T, color="tab:blue", alpha=0.5, label="HDI 50%")
    ax.set(title=title)

    ax.legend()
    if show:
        plt.show()

### Via posterior sampling of the shock matrix

In [ ]:
steps = 40
irf = ss_mod.impulse_response_function(idata, n_steps=steps, orthogonalize_shocks=True)
plot_irf(irf.isel(state=0), "Impulse response function from estimated covariance matrix")

### Create a custom shock scenario

In [ ]:
shock_trajectory = np.zeros((steps, ss_mod.k_posdef))
shock_trajectory[0] = 1
shock_trajectory[20] = 0.5

irf = ss_mod.impulse_response_function(idata, shock_trajectory=shock_trajectory)
plot_irf(irf.isel(state=0), "Impulse response function with shock of 1 at t=0 and 0.5 at t=20")

## Interpretable ARMA

In addition to the usual formulation, there's also an "interpretable" formulation of the ARMA model. It has significantly more states, which makes it a bad choice if speed is critical. But for a smallish model on smallish data, it lets us directly recover the innovation trajectory.

We can also add measurement error to the model, in case we have some reason to doubt the data. We'll also show off the missing data interpolation ability of the state space model.

In [ ]:
ss_mod = pmss.BayesianSARIMAX(
    order=(1, 0, 1),
    state_structure="interpretable",
    measurement_error=True,
    verbose=True,
)

In [ ]:
with pm.Model(coords=ss_mod.coords) as arma_model:
    state_sigmas = pm.Gamma(
        "sigma_state", alpha=2.0, beta=1.0, dims=ss_mod.param_dims.get("sigma_state")
    )
    obs_sigmas = pm.Gamma("sigma_obs", alpha=2.0, beta=1.0, dims=ss_mod.param_dims.get("sigma_obs"))
    rho = pm.TruncatedNormal(
        "ar_params",
        mu=0.0,
        sigma=0.5,
        lower=-1.0,
        upper=1.0,
        dims=ss_mod.param_dims.get("ar_params"),
    )
    theta = pm.Normal("ma_params", mu=0.0, sigma=0.5, dims=ss_mod.param_dims.get("ma_params"))
    ss_mod.build_statespace_graph(df)

    idata = pm.sample(**sample_kwargs())

In [ ]:
az.plot_trace_dist(idata, var_names=list(ss_mod.param_names))
plt.tight_layout()

In [ ]:
post_pred = ss_mod.sample_conditional_posterior(idata)

We can see that the smoother output for the observed Data variable is no longer a perfect fit and has some posterior uncertainty resulting from measurement error. The hidden state is now the exact innovation series, unscaled by theta or lagged, which makes it easier to interpret, as advertised.

Also, since we are trusting the data less, the Smoother learns a lot less about the hidden state relative to the Filter -- the two are very similar.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 6), dpi=144)
post = az.extract(post_pred, "posterior_predictive")
colors = ["tab:" + c for c in ["blue", "orange", "green"]]

for filter_output, color in zip(["filtered", "predicted", "smoothed"], colors):
    hdi = az.hdi(post_pred.to_dataset())[f"{filter_output}_posterior"]
    for idx, (axis, name) in enumerate(
        zip(fig.axes, ["Observed State (Data)", "Hidden State (ARMA Dynamics)"])
    ):
        post[f"{filter_output}_posterior"].isel(state=idx).mean(dim="sample").plot.line(
            x="time", ax=axis, lw=2, add_legend=False, label=filter_output.title(), color=color
        )
        axis.fill_between(
            hdi.coords["time"], *hdi.isel(state=idx).values.T, alpha=0.25, color=color
        )
        axis.set(title=name)

ax[0].legend()
fig.set_facecolor("w")
fig.tight_layout()
plt.show()

Use an IRF to illustrate that the innovations state really is just a record of the shocks

In [ ]:
irf = ss_mod.impulse_response_function(idata, shock_trajectory=shock_trajectory)
fig, ax = plt.subplots(2, 1)
plot_irf(irf.sel(state="data"), title="", ax=ax[0], show=False)
plot_irf(irf.sel(state="innovations"), title="", ax=ax[1])

# Seasonal Terms and Differences

Next, consider the airline passenger dataset. This dataset is interesting for several reasons. It has a non-stationary trends, and exhibits a strong seasonal pattern.

In [ ]:
airpass = pd.read_csv(
    "../tests/statespace/_data/airpass.csv",
    parse_dates=True,
    date_format="%Y %b",
    index_col=0,
    dtype={"value": float},
).rename(columns={"value": "passengers"})
airpass.index.freq = airpass.index.inferred_freq
airpass.plot()

SARIMA models require stationary data to be applied. This data is actually non-stationary in two ways. First, there is the trend. This can be handled via differencing. In many packages, differencing the data will force you to drop datapoints, but `statespace.BayesianSARIMA` can do the differencing inside the state space automatically, without losing any data.

The second source of non-stationarity is the growing autocovariance. It is clear that the size of the seasonal pattern is increasing over time, which is a tell-tale sign of a multiplicative timeseries, wherein the level of the series gets into the variance. Unfortunately, linear state space models can only handle linear models. So we will need to convert the multiplicative errors to linear errors by taking logs of the data. This cannot be done inside the model, and needs to be done by hand.

Let's take a look at what the log differences look like. This is just for illustration -- as noted, we don't actually need to difference by hand

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
airpass.apply(np.log).plot(ax=ax[0], legend=False, title="Log Passengers")
airpass.apply(np.log).diff().plot(ax=ax[1], legend=False, title="D1.Log Passengers")
ax[1].set_xlabel("Time")
plt.show()

This looks stationary. There is no longer any upward trend, and the seasonal pattern looks stable over time. We could formalize the finding with an ADF or KPSS test, but we'll just go with the eyeball test.

Next we need to choose a seasonal order. It's annual data, so 12 seems reasonable. For the ARIMA order, we will use (2, 1, 2), and for the seasonal order, (2, 0, 2, 12).

In [ ]:
ss_mod = pmss.BayesianSARIMAX(
    order=(2, 1, 2),
    seasonal_order=(2, 0, 2, 12),
    verbose=True,
    stationary_initialization=False,
)

In [ ]:
with pm.Model(coords=ss_mod.coords) as arma_model:
    # State 0 is associated with the observed data, and has a non-zero initial state (because the data has a non-zero intercept)
    intercept = pm.Normal("intercept", mu=4.5, sigma=1, shape=(1,))
    x0 = pm.Deterministic(
        "x0", pt.concatenate([intercept, pt.zeros(ss_mod.k_states - 1)]), dims=["state"]
    )

    # Give State 0 (the non-zero one) its own sigma for the initial covariance, while all the stationary states can share a single
    # sigma
    sigma_P0 = pm.Gamma("sigma_P0", alpha=2, beta=10, shape=(2,))
    P0 = pt.eye(ss_mod.k_states) * sigma_P0[1]
    P0 = pt.set_subtensor(P0[0, 0], sigma_P0[0])
    P0 = pm.Deterministic("P0", P0, dims=["state", "state_aux"])

    ar_params = pm.Normal("ar_params", mu=0.0, sigma=0.5, dims=ss_mod.param_dims.get("ar_params"))
    seasonal_ar_params = pm.Normal(
        "seasonal_ar_params", mu=0.0, sigma=0.5, dims=ss_mod.param_dims.get("seasonal_ar_params")
    )

    ma_params = pm.Normal("ma_params", mu=0.0, sigma=0.5, dims=ss_mod.param_dims.get("ma_params"))
    seasonal_ma_params = pm.Normal(
        "seasonal_ma_params", mu=0.0, sigma=0.5, dims=ss_mod.param_dims.get("seasonal_ma_params")
    )

    state_sigmas = pm.Gamma(
        "sigma_state", alpha=2, beta=1.0, dims=ss_mod.param_dims.get("sigma_state")
    )

    # Remember to log the data by hand
    ss_mod.build_statespace_graph(airpass.apply(np.log))

    idata = pm.sample(**sample_kwargs())

In [ ]:
az.plot_trace_dist(idata, var_names=[rv.name for rv in arma_model.basic_RVs[:-1]])

### Conditional Posterior

In [ ]:
post_pred = ss_mod.sample_conditional_posterior(idata)

In [ ]:
fig, ax = plt.subplots()
post = az.extract(post_pred, "posterior_predictive").map(np.exp)
hdi = az.hdi(post_pred.to_dataset().map(np.exp))["predicted_posterior_observed"]
post["predicted_posterior_observed"].isel(observed_state=0).mean(dim="sample").plot.line(
    x="time", ax=ax, add_legend=False, label="Posterior Mean, Predicted"
)
ax.fill_between(
    hdi.coords["time"],
    *hdi.isel(observed_state=0).values.T,
    alpha=0.25,
    color="tab:blue",
    label="94% HDI",
)
ax.plot(airpass.index, airpass.values, label="Data")
ax.legend()
plt.show()

### Forecasts

In [ ]:
forecast_idata = ss_mod.forecast(idata, start=airpass.index[-1], periods=20)

In [ ]:
forecast_hdi = az.hdi(forecast_idata.to_dataset().map(np.exp)).forecast_observed.isel(
    observed_state=0
)
forecast_mu = forecast_idata.to_dataset().map(np.exp).forecast_observed.mean(dim=["chain", "draw"])
fig, ax = plt.subplots()
ax.plot(airpass.index, airpass.values, label="Data")
ax.plot(forecast_mu.coords["time"], forecast_mu, label="Mean Forecast")
ax.fill_between(
    forecast_mu.coords["time"],
    *forecast_hdi.values.T,
    label="Forecast 94% HDI",
    color="tab:orange",
    alpha=0.25,
)
ax.legend()
plt.show()